# Day 7 - Advanced NLP and Generative Models Assignment
**Student Name:** Supriya Misal

This portable version is designed to run in **Google Colab, Jupyter Notebook, or JupyterLab**.
It automatically finds `Sentiment.csv`/`Sentiment.csv.zip` in the notebook folder, `/content`, or `/mnt/data`.


In [ ]:
# Step 1: Portable setup and dataset discovery
# ---------------------------------------------------------------
# This cell works on Windows/Linux/macOS, Jupyter, JupyterLab, and Colab.
# ---------------------------------------------------------------

from pathlib import Path
import zipfile
import sys
import subprocess

print("Python:", sys.version)

# Locations commonly used by Jupyter and Google Colab
SEARCH_DIRS = [
    Path.cwd(),
    Path("/content"),
    Path("/content/sample_data"),
    Path("/mnt/data"),
]

# Remove duplicates while preserving order
SEARCH_DIRS = list(dict.fromkeys(p for p in SEARCH_DIRS if p.exists()))

def find_file(filename):
    """Find a file in common notebook/data locations."""
    for base in SEARCH_DIRS:
        direct = base / filename
        if direct.exists():
            return direct
        try:
            matches = list(base.rglob(filename))
            if matches:
                return matches[0]
        except (PermissionError, OSError):
            pass
    return None

# Look for an existing CSV first
csv_path = find_file("Sentiment.csv")

# If only the ZIP exists, extract it next to the ZIP
if csv_path is None:
    zip_path = find_file("Sentiment.csv.zip")
    if zip_path is not None:
        extract_dir = zip_path.parent / "sentiment_data"
        extract_dir.mkdir(exist_ok=True)
        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(extract_dir)
        csv_path = find_file("Sentiment.csv")
        if csv_path is None:
            # Handle ZIPs whose CSV has a slightly different path/name
            csv_candidates = list(extract_dir.rglob("*.csv"))
            if csv_candidates:
                csv_path = csv_candidates[0]

if csv_path is not None:
    print(f"Dataset found: {csv_path}")
else:
    print("Sentiment.csv was not found.")
    print("The next cell will create a small demo dataset so the notebook can still run.")
    print("For the assignment results, place your real Sentiment.csv (or Sentiment.csv.zip)")
    print("in the same folder as this notebook and run all cells again.")

print("Student: Supriya Misal")


In [ ]:
# Step 2: Install/import required packages and load the dataset
# ---------------------------------------------------------------
# Missing Python packages are installed automatically.
# ---------------------------------------------------------------

import importlib.util
import subprocess
import sys

required = {
    "pandas": "pandas",
    "transformers": "transformers",
    "sklearn": "scikit-learn",
    "torch": "torch",
}

missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Installing missing packages:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Load the real dataset if available.
if csv_path is not None:
    df = pd.read_csv(csv_path)
else:
    # Small fallback only to verify that the notebook executes end-to-end.
    demo_data = {
        "text": [
            "I absolutely loved this product.",
            "This was a wonderful experience.",
            "The service was excellent and very helpful.",
            "I am very happy with the result.",
            "This is terrible and disappointing.",
            "I hated the experience.",
            "The product was poor and useless.",
            "I am unhappy with this purchase.",
        ],
        "sentiment": [
            "positive", "positive", "positive", "positive",
            "negative", "negative", "negative", "negative"
        ],
    }
    df = pd.DataFrame(demo_data)
    print("WARNING: Using the built-in demo dataset, not the assignment dataset.")

print("Dataset shape:", df.shape)
display(df.head())
print("\nAvailable columns:", df.columns.tolist())

# Identify the text column
preferred_text_columns = ["text", "tweet", "review", "sentence", "content"]
text_column = next((c for c in preferred_text_columns if c in df.columns), None)

if text_column is None:
    object_columns = df.select_dtypes(include="object").columns.tolist()
    # Prefer a column with text-like values
    if object_columns:
        text_column = max(
            object_columns,
            key=lambda c: df[c].fillna("").astype(str).str.len().mean()
        )

if text_column is None:
    raise ValueError(
        "No text column was found. Expected a column such as 'text', 'tweet', "
        "'review', 'sentence', or 'content'."
    )

print(f"Using '{text_column}' as text column.")
texts = df[text_column].fillna("").astype(str).tolist()


In [ ]:
# Step 3: Load the Hugging Face sentiment pipeline
# ---------------------------------------------------------------
# The first run may download the model from Hugging Face.
# An internet connection is required only for that first model download.
# ---------------------------------------------------------------

MODEL_NAME = "distilbert-base-uncased-finetuned-sst-2-english"

try:
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model=MODEL_NAME
    )
except Exception as exc:
    raise RuntimeError(
        "\nCould not load the Hugging Face model.\n"
        "Please check your internet connection and run this cell again. "
        "After the model is downloaded once, it can normally be reused from "
        "the local Hugging Face cache.\n\n"
        f"Original error: {exc}"
    ) from exc

# Batch size 8 as used in the original assignment.
batch_size = 8

results = sentiment_pipeline(
    texts,
    truncation=True,
    max_length=192,
    batch_size=batch_size
)

print(f"Generated {len(results)} sentiment predictions.")


In [ ]:
# Step 4: Store predictions and confidence scores
# ---------------------------------------------------------------

df["predicted_label"] = [r["label"] for r in results]
df["confidence_score"] = [round(float(r["score"]), 4) for r in results]

display(df[[text_column, "predicted_label", "confidence_score"]].head(10))


In [ ]:
# Step 5: Confidence analysis
# ---------------------------------------------------------------

high_confidence = df[df["confidence_score"] >= 0.80]
low_confidence = df[df["confidence_score"] < 0.80]

print("\nTotal predictions:", len(df))
print("High-confidence predictions (>= 80%):", len(high_confidence))
print("Lower-confidence predictions (< 80%):", len(low_confidence))
print("Average confidence:", f"{df['confidence_score'].mean():.2%}")

print("\nPrediction distribution:")
print(df["predicted_label"].value_counts())


In [ ]:
# Step 6: Evaluate against available sentiment labels
# ---------------------------------------------------------------

if "sentiment" in df.columns:
    actual = df["sentiment"].astype(str).str.strip().str.lower()
    predicted = df["predicted_label"].astype(str).str.strip().str.lower()

    valid = actual.isin(["positive", "negative"])

    if valid.any():
        accuracy = accuracy_score(actual[valid], predicted[valid])
        print(f"\nAccuracy: {accuracy:.2%}")

        print("\nClassification report:")
        print(classification_report(
            actual[valid],
            predicted[valid],
            labels=["negative", "positive"],
            zero_division=0
        ))

        print("Confusion matrix:")
        print(confusion_matrix(
            actual[valid],
            predicted[valid],
            labels=["negative", "positive"]
        ))
    else:
        print("\nNo positive/negative ground-truth labels available for evaluation.")
else:
    print("\nNo sentiment column available for evaluation.")


In [ ]:
# Step 7: Save the modified results
# ---------------------------------------------------------------

output_file = Path.cwd() / "Supriya_Misal_Sentiment_with_predictions.csv"
df.to_csv(output_file, index=False)

print(f"\nResults saved to: {output_file.resolve()}")
print("Notebook completed successfully.")
